In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from src.models.rnn import RNN
from src.models.gru import GRU
from src.models.lstm import LSTM
from src.data_loader import load_symbol
from src.features import build_features

SEED = 42
L = 30
HIDDEN = 32

In [12]:
for M in (RNN, GRU, LSTM):
    print(M.__name__, M(input_size=8, hidden_size=16, output_size=1)(torch.randn(32, 20, 8)).shape)

RNN torch.Size([32, 1])
GRU torch.Size([32, 1])
LSTM torch.Size([32, 1])


In [13]:
es_df = load_symbol('ES=F', start='2016-01-01', end='2022-01-01')
es_features = build_features(es_df)

es_features['y'] = es_features['log_return'].shift(-1)
es_features = es_features.replace([np.inf, -np.inf], np.nan).dropna()

y = es_features['y']
X = es_features.drop(columns=['y'])

n = len(X)
i_val = int(n * 0.60)
i_test = int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i_val], X.iloc[i_val:i_test], X.iloc[i_test:]
y_train, y_val, y_test = y.iloc[:i_val], y.iloc[i_val:i_test], y.iloc[i_test:]

sx = StandardScaler().fit(X_train)
sy = StandardScaler().fit(y_train.values.reshape(-1, 1))

def make_windows(X_part, y_part, L):
    Xs = sx.transform(X_part)
    ys = sy.transform(y_part.values.reshape(-1, 1)).ravel()
    xw, yw = [], []
    for i in range(L, len(Xs) + 1):
        xw.append(Xs[i - L:i])
        yw.append(ys[i - 1])
    xw = torch.tensor(np.array(xw), dtype=torch.float32)
    yw = torch.tensor(np.array(yw), dtype=torch.float32).unsqueeze(1)
    return xw, yw

Xtr, ytr = make_windows(X_train, y_train, L)
Xva, yva = make_windows(X_val, y_val, L)
Xte, yte = make_windows(X_test, y_test, L)
print(Xtr.shape, Xva.shape, Xte.shape)

torch.Size([861, 30, 16]) torch.Size([342, 30, 16]) torch.Size([194, 30, 16])


/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/Users/sergheibi

In [14]:
def train_model(model_cls, epochs=100, patience=8, lr=1e-3, wd=1e-4, bs=64):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = model_cls(input_size=Xtr.shape[2], hidden_size=HIDDEN, output_size=1)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True)
    best_val, best_state, bad = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva), yva).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    print(f'{model_cls.__name__}: stopped at epoch {epoch}, best val {best_val:.4f}')
    return model

def evaluate(model, Xs, ys):
    model.eval()
    with torch.no_grad():
        p = model(Xs).numpy().ravel()
    t = ys.numpy().ravel()
    ic = np.corrcoef(p, t)[0, 1] if p.std() > 0 else np.nan
    return {
        'mse': np.mean((p - t) ** 2),
        'ic': ic,
        'hit': np.mean(np.sign(p) == np.sign(t)),
        'pred_std': p.std(),
    }

models = {cls.__name__: train_model(cls) for cls in (RNN, GRU, LSTM)}

RNN: stopped at epoch 12, best val 5.1217
GRU: stopped at epoch 12, best val 5.1033
LSTM: stopped at epoch 8, best val 5.1642


In [15]:
splits = {'train': (Xtr, ytr), 'val': (Xva, yva), 'test': (Xte, yte)}
rows = []
for name, m in models.items():
    for sp, (Xs, ys) in splits.items():
        rows.append({'model': name, 'split': sp, **evaluate(m, Xs, ys)})

ytr_mean = ytr.mean().item()
for sp, (Xs, ys) in splits.items():
    t = ys.numpy().ravel()
    rows.append({'model': 'zero', 'split': sp, 'mse': np.mean(t ** 2), 'ic': np.nan, 'hit': np.nan, 'pred_std': 0.0})
    rows.append({'model': 'train_mean', 'split': sp, 'mse': np.mean((t - ytr_mean) ** 2), 'ic': np.nan, 'hit': np.nan, 'pred_std': 0.0})
    rows.append({'model': 'always_up', 'split': sp, 'mse': np.nan, 'ic': np.nan, 'hit': np.mean(t > 0), 'pred_std': 0.0})

res = pd.DataFrame(rows).set_index(['split', 'model']).sort_index()
res.round(4)

mse      ic     hit  pred_std
split model                                       
test  GRU         0.8453 -0.0162  0.5000    0.1225
      LSTM        0.8660 -0.0408  0.4536    0.0483
      RNN         0.9920  0.0293  0.4536    0.0711
      always_up      NaN     NaN  0.5464    0.0000
      train_mean  0.8289     NaN     NaN    0.0000
      zero        0.8275     NaN     NaN    0.0000
train GRU         0.9594  0.1932  0.5436    0.1177
      LSTM        0.9792  0.1210  0.5134    0.0666
      RNN         0.9626  0.2193  0.5319    0.0789
      always_up      NaN     NaN  0.4959    0.0000
      train_mean  0.9908     NaN     NaN    0.0000
      zero        0.9909     NaN     NaN    0.0000
val   GRU         5.1033  0.0666  0.4854    0.1695
      LSTM        5.1642 -0.0624  0.4386    0.1006
      RNN         5.1217  0.0714  0.4561    0.1520
      always_up      NaN     NaN  0.5643    0.0000
      train_mean  5.1166     NaN     NaN    0.0000
      zero        5.1155     NaN     NaN    0.0000